# Procesamiento y validación de los Conteos Censales de Participación Ciudadana 2012

**Proyecto:** Hackea la participación: Datos que construyen democracia  
**Reto 1:** Perfiles ciudadanos de participación  
**Etapa:** Homologación histórica 2009–2024  
**Año procesado:** 2012

## Objetivo

Este notebook procesa la base oficial de los **Conteos Censales de Participación Ciudadana 2012** del Instituto Nacional Electoral (INE) y la transforma a la misma unidad de análisis utilizada para 2024:

\[
\text{Entidad} + \text{Grupo de edad} + \text{Sexo} + \text{Tipo de sección}
\]

El propósito es obtener una base agregada y validada que pueda integrarse posteriormente con los demás años para construir el historial de participación utilizado en la segmentación de perfiles ciudadanos.

### Principios de trabajo

- Se conservan los datos originales sin modificación.
- Se documentan diferencias de estructura respecto de 2024.
- La participación se calcula **después de agregar los conteos**, no como promedio simple de porcentajes.
- Los grupos pequeños se conservan y únicamente se marcan para análisis de sensibilidad.
- Si una variable esperada no existe o tiene una codificación distinta, el notebook lo reporta antes de continuar.

## Fuente oficial

La base 2012 forma parte de los Conteos Censales de Participación Ciudadana publicados por el INE.  
La descarga utilizada corresponde al archivo oficial:

`DatosAbiertos_DECEyEC_ConteosCensales2012.zip`

El archivo ZIP se carga manualmente en Google Colab para evitar respuestas HTML o descargas incompletas del servidor.

> **Nota de reproducibilidad:** los archivos descargados o subidos directamente a Colab son temporales y desaparecen al reiniciar el entorno.

### Precaución para 2012

Aunque el pipeline es el mismo que en los años posteriores, **no se asumirá que la estructura de 2012 es idéntica**. El notebook primero inspecciona el diccionario y una muestra de la base. Si falta alguna variable obligatoria o aparece una codificación distinta en `SEXO`, `TIPOSEC`, `NS` o cualquier otra variable necesaria, el proceso se detendrá para revisarla antes de continuar.

La fuente utilizada es la descarga oficial de 2012 enlazada desde la página de Conteos Censales del INE.

In [20]:
import os
import io
import re
import zipfile
import pandas as pd
import requests
import numpy as np

pd.set_option("display.max_columns", None)

## 1. Obtención del archivo ZIP

Para facilitar la reproducibilidad, el notebook intenta descargar automáticamente la base oficial de **2012** desde el sitio del INE.

Antes de continuar, la descarga se valida con `zipfile.is_zipfile()`. Si el servidor devuelve una página HTML, un archivo incompleto o cualquier contenido que no sea un ZIP válido, el notebook solicita subir manualmente el archivo oficial.

> **Nota:** los archivos almacenados en `/content/` son temporales y desaparecen al reiniciar el entorno de Google Colab.

In [21]:
URL_2012 = "https://www.ine.mx/wp-content/uploads/2022/04/DatosAbiertos_DECEyEC_ConteosCensales2012.zip"
RUTA_ZIP = "/content/DatosAbiertos_DECEyEC_ConteosCensales2012.zip"

def descargar_zip_oficial(url, destino):
    print("Descargando base oficial desde el sitio del INE...")

    respuesta = requests.get(
        url,
        timeout=120,
        allow_redirects=True,
        headers={"User-Agent": "Mozilla/5.0"}
    )
    respuesta.raise_for_status()

    with open(destino, "wb") as f:
        f.write(respuesta.content)

    return zipfile.is_zipfile(destino)

# Si existe un archivo con el nombre esperado pero no es un ZIP válido,
# se elimina para evitar reutilizar una descarga incompleta o HTML.
if os.path.exists(RUTA_ZIP) and not zipfile.is_zipfile(RUTA_ZIP):
    os.remove(RUTA_ZIP)

zip_valido = os.path.exists(RUTA_ZIP) and zipfile.is_zipfile(RUTA_ZIP)

if zip_valido:
    print("Se encontró un ZIP válido en /content/:", RUTA_ZIP)
else:
    try:
        zip_valido = descargar_zip_oficial(URL_2012, RUTA_ZIP)
    except Exception as e:
        print("La descarga automática produjo un error:", e)
        zip_valido = False

if not zip_valido:
    print("\nLa descarga automática no produjo un ZIP válido.")
    print("Sube manualmente el archivo oficial de 2012.")

    try:
        from google.colab import files
        subidos = files.upload()

        if not subidos:
            raise RuntimeError("No se cargó ningún archivo.")

        nombre_subido = next(iter(subidos))
        contenido = subidos[nombre_subido]

        with open(RUTA_ZIP, "wb") as f:
            f.write(contenido)

    except ImportError:
        raise RuntimeError(
            "Carga manualmente el ZIP de 2012 en /content/ "
            "y vuelve a ejecutar esta celda."
        )

if not zipfile.is_zipfile(RUTA_ZIP):
    raise ValueError(
        "El archivo disponible no es un ZIP válido. "
        "Vuelve a descargar la base oficial desde el INE."
    )

print("Archivo listo:", RUTA_ZIP)
print("ZIP válido: True")

Se encontró un ZIP válido en /content/: /content/DatosAbiertos_DECEyEC_ConteosCensales2012.zip
Archivo listo: /content/DatosAbiertos_DECEyEC_ConteosCensales2012.zip
ZIP válido: True


## 2. Inventario del ZIP

Se identifican los archivos CSV disponibles y, cuando existe, el archivo de descripción de campos incluido por el INE.

In [22]:
with zipfile.ZipFile(RUTA_ZIP, "r") as z:
    archivos = z.namelist()

archivos_csv = [a for a in archivos if a.lower().endswith(".csv")]
archivos_txt = [a for a in archivos if a.lower().endswith(".txt")]

print("Archivos totales:", len(archivos))
print("Archivos CSV:", len(archivos_csv))
print("Archivos TXT:", len(archivos_txt))

print("\nPrimeros archivos:")
for a in archivos[:10]:
    print(" -", a)

if len(archivos_csv) != 32:
    raise ValueError(
        f"Se esperaban 32 archivos CSV para 2012, pero se encontraron {len(archivos_csv)}."
    )

print("Inventario validado: 32 archivos estatales.")

Archivos totales: 34
Archivos CSV: 32
Archivos TXT: 1

Primeros archivos:
 - DatosAbiertos_DECEyEC_ConteosCensales2012/
 - DatosAbiertos_DECEyEC_ConteosCensales2012/01_descripcioncampos_conteoscensales2012.txt
 - DatosAbiertos_DECEyEC_ConteosCensales2012/datosabiertos_deceyec_conteoscensales2012_ags.csv
 - DatosAbiertos_DECEyEC_ConteosCensales2012/datosabiertos_deceyec_conteoscensales2012_bc.csv
 - DatosAbiertos_DECEyEC_ConteosCensales2012/datosabiertos_deceyec_conteoscensales2012_bcs.csv
 - DatosAbiertos_DECEyEC_ConteosCensales2012/datosabiertos_deceyec_conteoscensales2012_camp.csv
 - DatosAbiertos_DECEyEC_ConteosCensales2012/datosabiertos_deceyec_conteoscensales2012_cdmx.csv
 - DatosAbiertos_DECEyEC_ConteosCensales2012/datosabiertos_deceyec_conteoscensales2012_chih.csv
 - DatosAbiertos_DECEyEC_ConteosCensales2012/datosabiertos_deceyec_conteoscensales2012_chis.csv
 - DatosAbiertos_DECEyEC_ConteosCensales2012/datosabiertos_deceyec_conteoscensales2012_coah.csv
Inventario validado: 32 ar

### Archivo de descripción

Cuando el ZIP contiene un archivo `.txt`, se muestran sus primeras líneas.  
Este archivo debe considerarse la referencia principal para interpretar la codificación específica de 2012.

In [23]:
if archivos_txt:
    with zipfile.ZipFile(RUTA_ZIP, "r") as z:
        contenido = z.read(archivos_txt[0])

    texto = None
    for enc in ["utf-8", "latin-1", "cp1252"]:
        try:
            texto = contenido.decode(enc)
            break
        except UnicodeDecodeError:
            pass

    print("\n".join(texto.splitlines()[:80]))
else:
    print("No se encontró archivo TXT de descripción dentro del ZIP.")

Descripción archivo
AELEC	Año de elección (2009,2012,2015, 2018 y 2021)
FELECCION	Fecha de la jornada electoral (formato corto)
EDOCVE	Clave INE de entidad federativa
EDONOM	Nombre de la entidad federativa
MPIOCVE	Clave INE de Municipio
MPIONOM	Nombre del municipio
SECCION	Sección electoral en el año correspondiente
TIPOSEC	Tipo de sección en el año correspondiente (Urbana, Mixta, Rural)
DEF	Distrito Electoral Federal en el marco cartográfico de la DERFE 2021
DEL	Distrito Electoral Local en el marco cartográfico de la DERFE 2021
SEXO	Sexo de las personas (0 Hombres; 1 Mujeres)
EDAD	Años de edad cumplidos al día de la elección de las personas
LN	Lista nominal de los cuadernillos para todas las casillas catalogadas
SV	Registros con marca de voto.
NV	Registros sin marca de voto.
NS	Registros no especificados en participación, pues corresponden a:
	- Registro correspondiente a un cuadernillo sin marcas de votación.
	- Registro correspondiente a un cuadernillo no disponible para su captura.

## 3. Función de lectura robusta

Algunas bases históricas pueden utilizar codificaciones de texto distintas.  
La siguiente función intenta leer cada CSV con varias codificaciones comunes, sin alterar los datos.

In [24]:
def leer_csv_desde_zip(z, nombre):
    ultimo_error = None

    for encoding in ["utf-8", "latin-1", "cp1252"]:
        try:
            with z.open(nombre) as f:
                return pd.read_csv(
                    f,
                    encoding=encoding,
                    low_memory=False
                    )
        except UnicodeDecodeError as e:
            ultimo_error = e

    raise ultimo_error

## 4. Inspección de un archivo estatal

Antes de procesar todo el país se revisa un CSV para conocer las columnas y algunos valores de ejemplo.

In [25]:
with zipfile.ZipFile(RUTA_ZIP, "r") as z:
    df_muestra = leer_csv_desde_zip(z, archivos_csv[0])

print("Archivo de muestra:", archivos_csv[0])
print("Dimensiones:", df_muestra.shape)
display(df_muestra.head())
print("\nColumnas:")
print(df_muestra.columns.tolist())

if "AELEC" not in df_muestra.columns:
    raise ValueError("La base no contiene la columna AELEC para verificar el año.")

anios_muestra = sorted(
    pd.to_numeric(df_muestra["AELEC"], errors="coerce")
      .dropna()
      .astype(int)
      .unique()
      .tolist()
)

if anios_muestra != [2012]:
    raise ValueError(
        f"El archivo cargado no corresponde exclusivamente a 2012. "
        f"Años detectados: {anios_muestra}"
    )

print("Año verificado:", anios_muestra[0])

Archivo de muestra: DatosAbiertos_DECEyEC_ConteosCensales2012/datosabiertos_deceyec_conteoscensales2012_ags.csv
Dimensiones: (77141, 16)


,AELEC,FELECCION,EDOCVE,EDONOM,MPIOCVE,MPIONOM,SECCION,TIPOSEC,DEF,DEL,SEXO,EDAD,LN,SV,NV,NS
0,2012,1/7/2012,1,AGUASCALIENTES,1,Aguascalientes,1,U,3,6,0,18,27,21,6,0
1,2012,1/7/2012,1,AGUASCALIENTES,1,Aguascalientes,1,U,3,6,0,19,30,19,11,0
2,2012,1/7/2012,1,AGUASCALIENTES,1,Aguascalientes,1,U,3,6,0,20,24,18,6,0
3,2012,1/7/2012,1,AGUASCALIENTES,1,Aguascalientes,1,U,3,6,0,21,37,24,13,0
4,2012,1/7/2012,1,AGUASCALIENTES,1,Aguascalientes,1,U,3,6,0,22,20,12,8,0



Columnas:
['AELEC', 'FELECCION', 'EDOCVE', 'EDONOM', 'MPIOCVE', 'MPIONOM', 'SECCION', 'TIPOSEC', 'DEF', 'DEL', 'SEXO', 'EDAD', 'LN', 'SV', 'NV', 'NS']
Año verificado: 2012


## 5. Compatibilidad estructural con el procesamiento 2024

Para mantener una metodología común entre años, se revisan las variables necesarias para la unidad de análisis y los indicadores.

Las variables mínimas esperadas son:

- `EDONOM`: entidad federativa.
- `EDAD`: edad.
- `SEXO`: sexo.
- `TIPOSEC`: tipo de sección.
- `LN`: lista nominal.
- `SV`: sí votaron.
- `NV`: no votaron.

`NS` se utiliza cuando existe para representar registros con situación de participación no especificada.

> Si `NS` no está presente, el notebook solo asumirá `NS = 0` cuando pueda comprobarse que `LN = SV + NV` en todos los registros.

### Precaución para 2012

A partir de este año pueden aparecer diferencias mayores respecto de 2018, 2021 y 2024 en nombres de columnas, categorías o cobertura de variables. Por ello, este notebook **no fuerza equivalencias automáticamente**: si detecta una variable obligatoria ausente o un código no reconocido, se detiene para que la diferencia sea revisada y documentada.

Esto es importante para evitar introducir supuestos en la homologación histórica.

In [26]:
columnas_esperadas = ["EDONOM", "EDAD", "SEXO", "TIPOSEC", "LN", "SV", "NV", "NS"]

compatibilidad = pd.DataFrame({
    "VARIABLE": columnas_esperadas,
    "PRESENTE_2012": [c in df_muestra.columns for c in columnas_esperadas]
})

display(compatibilidad)

obligatorias = ["EDONOM", "EDAD", "SEXO", "TIPOSEC", "LN", "SV", "NV"]
faltantes = [c for c in obligatorias if c not in df_muestra.columns]

if faltantes:
    raise ValueError(
        "Faltan columnas obligatorias para aplicar el pipeline: "
        + ", ".join(faltantes)
    )

print("Variables obligatorias disponibles.")

,VARIABLE,PRESENTE_2012
0,EDONOM,True
1,EDAD,True
2,SEXO,True
3,TIPOSEC,True
4,LN,True
5,SV,True
6,NV,True
7,NS,True


Variables obligatorias disponibles.


## 6. Preparación metodológica

Se utilizan los mismos grupos de edad definidos para 2024:

- 18–19
- 20–24
- 25–29
- ...
- 70–74
- 75+

El grupo `75+` evita que edades muy altas generen categorías con muy pocos registros y mantiene una clasificación interpretable.

La codificación descriptiva de `SEXO` y `TIPOSEC` se aplica únicamente a los códigos reconocidos.  
Cualquier código adicional queda identificado para revisión y no se reemplaza silenciosamente.

> **Sexo en las bases históricas:** para esta serie se utilizan los códigos `0 = Hombre` y `1 = Mujer`. Cualquier código adicional se considera desconocido y detiene el pipeline para revisión, en lugar de asignarse automáticamente.

In [27]:
BINS_EDAD = [
    18, 20, 25, 30, 35, 40, 45,
    50, 55, 60, 65, 70, 75, float("inf")
]

LABELS_EDAD = [
    "18-19", "20-24", "25-29", "30-34",
    "35-39", "40-44", "45-49", "50-54",
    "55-59", "60-64", "65-69", "70-74", "75+"
]

MAP_SEXO = {
    0: "Hombre",
    1: "Mujer"
}

MAP_TIPOSEC = {
    "U": "Urbana",
    "M": "Mixta",
    "R": "Rural"
}

## 7. Función de procesamiento por entidad

Para reducir el uso de memoria, cada archivo estatal se prepara y agrega antes de integrarse a la base nacional.

Este procedimiento produce el mismo tipo de unidad de análisis que en 2024, pero evita mantener simultáneamente millones de filas crudas en memoria.

In [28]:
def preparar_y_agrupar(df, nombre_archivo=""):
    df = df.copy()

    # Verificar que cada archivo estatal pertenezca al año esperado.
    if "AELEC" not in df.columns:
        raise ValueError(f"{nombre_archivo}: falta la columna AELEC.")

    anios_detectados = sorted(
        pd.to_numeric(df["AELEC"], errors="coerce")
          .dropna()
          .astype(int)
          .unique()
          .tolist()
    )

    if anios_detectados != [2012]:
        raise ValueError(
            f"{nombre_archivo}: años detectados {anios_detectados}; "
            f"se esperaba únicamente 2012."
        )

    # Convertir variables cuantitativas a numéricas.
    for col in ["EDAD", "LN", "SV", "NV"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Manejo responsable de NS.
    if "NS" not in df.columns:
        prueba = (
            df["LN"].notna()
            & df["SV"].notna()
            & df["NV"].notna()
            & (df["LN"] == df["SV"] + df["NV"])
        )

        if prueba.all():
            df["NS"] = 0
            ns_inferido = True
        else:
            raise ValueError(
                f"{nombre_archivo}: no existe NS y LN != SV + NV en algunos registros. "
                "Debe revisarse el diccionario de variables antes de continuar."
            )
    else:
        df["NS"] = pd.to_numeric(df["NS"], errors="coerce")
        ns_inferido = False

    # Revisar valores no numéricos en conteos.
    if df[["EDAD", "LN", "SV", "NV", "NS"]].isnull().any().any():
        filas_problematicas = df[
            df[["EDAD", "LN", "SV", "NV", "NS"]].isnull().any(axis=1)
        ]
        raise ValueError(
            f"{nombre_archivo}: existen valores no numéricos o faltantes "
            f"en variables esenciales ({len(filas_problematicas)} filas)."
        )

    # Grupos de edad.
    df["GRUPO_EDAD"] = pd.cut(
        df["EDAD"],
        bins=BINS_EDAD,
        labels=LABELS_EDAD,
        right=False
    )

    if df["GRUPO_EDAD"].isna().any():
        fuera_rango = df.loc[df["GRUPO_EDAD"].isna(), "EDAD"].unique()
        raise ValueError(
            f"{nombre_archivo}: hay edades fuera del esquema definido: {fuera_rango[:20]}"
        )

    # Descripciones.
    df["SEXO_DESC"] = df["SEXO"].map(MAP_SEXO)
    df["TIPOSEC_DESC"] = df["TIPOSEC"].map(MAP_TIPOSEC)

    # Detectar códigos no contemplados.
    codigos_sexo_desconocidos = sorted(
        df.loc[df["SEXO_DESC"].isna(), "SEXO"].dropna().unique().tolist()
    )
    codigos_tipo_desconocidos = sorted(
        df.loc[df["TIPOSEC_DESC"].isna(), "TIPOSEC"].dropna().unique().tolist()
    )

    if codigos_sexo_desconocidos:
        raise ValueError(
            f"{nombre_archivo}: códigos de SEXO no reconocidos: "
            f"{codigos_sexo_desconocidos}. Revisar diccionario 2012."
        )

    if codigos_tipo_desconocidos:
        raise ValueError(
            f"{nombre_archivo}: códigos de TIPOSEC no reconocidos: "
            f"{codigos_tipo_desconocidos}. Revisar diccionario 2012."
        )

    # Totales originales para validación.
    totales_originales = df[["LN", "SV", "NV", "NS"]].sum()

    # Comprobar consistencia LN = SV + NV + NS.
    consistencia = df["LN"] == (df["SV"] + df["NV"] + df["NS"])
    registros_inconsistentes = int((~consistencia).sum())

    # Agregación.
    agrupado = (
        df.groupby(
            ["EDONOM", "GRUPO_EDAD", "SEXO_DESC", "TIPOSEC_DESC"],
            observed=True,
            as_index=False
        )[["LN", "SV", "NV", "NS"]]
        .sum()
    )

    return agrupado, totales_originales, registros_inconsistentes, ns_inferido

## 8. Procesamiento de todos los archivos 2012

Cada CSV se procesa con el mismo procedimiento y posteriormente se integran los resultados nacionales.

In [ ]:
agrupados = []
totales_originales_2012 = pd.Series({"LN": 0, "SV": 0, "NV": 0, "NS": 0}, dtype="int64")
inconsistencias_totales = 0
archivos_con_ns_inferido = []

with zipfile.ZipFile(RUTA_ZIP, "r") as z:
    for i, nombre in enumerate(archivos_csv, start=1):
        df_estado = leer_csv_desde_zip(z, nombre)

        agrupado_estado, totales_estado, inconsistencias, ns_inferido = preparar_y_agrupar(
            df_estado,
            nombre_archivo=nombre
        )

        agrupados.append(agrupado_estado)
        totales_originales_2012 = totales_originales_2012.add(totales_estado, fill_value=0)
        inconsistencias_totales += inconsistencias

        if ns_inferido:
            archivos_con_ns_inferido.append(nombre)

        print(f"[{i}/{len(archivos_csv)}] Procesado: {nombre}")

df_2012_agrupado = (
    pd.concat(agrupados, ignore_index=True)
    .groupby(
        ["EDONOM", "GRUPO_EDAD", "SEXO_DESC", "TIPOSEC_DESC"],
        observed=True,
        as_index=False
    )[["LN", "SV", "NV", "NS"]]
    .sum()
)

print("\nProcesamiento nacional terminado.")
print("Dimensiones de la base agregada:", df_2012_agrupado.shape)
print("Entidades:", df_2012_agrupado["EDONOM"].nunique())
print("Registros inconsistentes LN = SV + NV + NS:", inconsistencias_totales)
print("Archivos en los que NS tuvo que inferirse:", len(archivos_con_ns_inferido))

[1/32] Procesado: DatosAbiertos_DECEyEC_ConteosCensales2012/datosabiertos_deceyec_conteoscensales2012_ags.csv
[2/32] Procesado: DatosAbiertos_DECEyEC_ConteosCensales2012/datosabiertos_deceyec_conteoscensales2012_bc.csv
[3/32] Procesado: DatosAbiertos_DECEyEC_ConteosCensales2012/datosabiertos_deceyec_conteoscensales2012_bcs.csv
[4/32] Procesado: DatosAbiertos_DECEyEC_ConteosCensales2012/datosabiertos_deceyec_conteoscensales2012_camp.csv
[5/32] Procesado: DatosAbiertos_DECEyEC_ConteosCensales2012/datosabiertos_deceyec_conteoscensales2012_cdmx.csv
[6/32] Procesado: DatosAbiertos_DECEyEC_ConteosCensales2012/datosabiertos_deceyec_conteoscensales2012_chih.csv
[7/32] Procesado: DatosAbiertos_DECEyEC_ConteosCensales2012/datosabiertos_deceyec_conteoscensales2012_chis.csv
[8/32] Procesado: DatosAbiertos_DECEyEC_ConteosCensales2012/datosabiertos_deceyec_conteoscensales2012_coah.csv
[9/32] Procesado: DatosAbiertos_DECEyEC_ConteosCensales2012/datosabiertos_deceyec_conteoscensales2012_col.csv
[10/32

## 9. Cálculo de indicadores

La participación se calcula sobre los registros con resultado determinado:

$$
\text{Participación} =
\frac{SV}{SV+NV}\times 100
$$

La abstención se calcula como:

$$
\text{Abstención} =
\frac{NV}{SV+NV}\times 100
$$

Los registros `NS` no se incorporan al denominador de estas dos tasas.

También se calcula su proporción respecto de la lista nominal para conservar un indicador de calidad de la información:

$$
\text{No especificado} =
\frac{NS}{LN}\times 100
$$


In [ ]:
df_2012_agrupado["N_DETERMINADO"] = (
    df_2012_agrupado["SV"] + df_2012_agrupado["NV"]
)

den = df_2012_agrupado["N_DETERMINADO"]

df_2012_agrupado["PARTICIPACION_PCT"] = np.where(
    den > 0,
    df_2012_agrupado["SV"] / den * 100,
    np.nan
)

df_2012_agrupado["ABSTENCION_PCT"] = np.where(
    den > 0,
    df_2012_agrupado["NV"] / den * 100,
    np.nan
)

df_2012_agrupado["NO_ESPECIFICADO_PCT"] = np.where(
    df_2012_agrupado["LN"] > 0,
    df_2012_agrupado["NS"] / df_2012_agrupado["LN"] * 100,
    np.nan
)

# Misma bandera provisional utilizada en 2024.
df_2012_agrupado["GRUPO_PEQUENO"] = (
    df_2012_agrupado["N_DETERMINADO"] < 100
)

display(df_2012_agrupado.head(20))

## 10. Validaciones de integridad

Las siguientes pruebas deben cumplirse antes de usar la base para el análisis histórico:

1. La agregación debe conservar los totales de `LN`, `SV`, `NV` y `NS`.
2. Deben aparecer las entidades disponibles en los archivos oficiales.
3. Participación + abstención debe ser aproximadamente 100 % cuando el denominador es mayor que cero.
4. Los grupos pequeños se conservan; no se eliminan en esta etapa.

In [ ]:
totales_agrupados_2012 = df_2012_agrupado[["LN", "SV", "NV", "NS"]].sum()

print("Totales originales:")
display(totales_originales_2012)

print("\nTotales después de agrupar:")
display(totales_agrupados_2012)

conserva_totales = np.allclose(
    totales_originales_2012.astype(float).values,
    totales_agrupados_2012.astype(float).values
)

print("\n¿Se conservaron los totales?:", conserva_totales)

In [ ]:
suma_tasas = (
    df_2012_agrupado["PARTICIPACION_PCT"]
    + df_2012_agrupado["ABSTENCION_PCT"]
)

print("Resumen de PARTICIPACION_PCT:")
display(df_2012_agrupado["PARTICIPACION_PCT"].describe())

print("\nParticipación + abstención:")
display(suma_tasas.describe())

print("\nGrupos pequeños:")
display(df_2012_agrupado["GRUPO_PEQUENO"].value_counts())

## 11. Resumen nacional descriptivo

El siguiente bloque calcula la participación nacional a partir de los **totales de conteos**, por lo que no debe confundirse con el promedio simple de los porcentajes de los grupos.

In [ ]:
totales = df_2012_agrupado[["LN", "SV", "NV", "NS"]].sum()
n_determinado = totales["SV"] + totales["NV"]

resumen_nacional_2012 = pd.Series({
    "LN": totales["LN"],
    "SV": totales["SV"],
    "NV": totales["NV"],
    "NS": totales["NS"],
    "N_DETERMINADO": n_determinado,
    "PARTICIPACION_PCT_NACIONAL": (
        totales["SV"] / n_determinado * 100
        if n_determinado > 0 else np.nan
    ),
    "ABSTENCION_PCT_NACIONAL": (
        totales["NV"] / n_determinado * 100
        if n_determinado > 0 else np.nan
    ),
    "NO_ESPECIFICADO_PCT_NACIONAL": (
        totales["NS"] / totales["LN"] * 100
        if totales["LN"] > 0 else np.nan
    )
})

display(resumen_nacional_2012)

## 12. Tabla de compatibilidad para la integración histórica

Esta tabla registra qué variables del esquema común están disponibles en 2012.  
Más adelante se combinará con las revisiones de 2009, 2012, 2012, 2012 y 2024.

In [ ]:
compatibilidad["OBSERVACION"] = ""

if "NS" not in df_muestra.columns:
    compatibilidad.loc[
        compatibilidad["VARIABLE"] == "NS",
        "OBSERVACION"
    ] = "No aparece en la muestra; el pipeline verifica si puede inferirse como 0."

compatibilidad["ANIO"] = 2012
compatibilidad = compatibilidad[["ANIO", "VARIABLE", "PRESENTE_2012", "OBSERVACION"]]

display(compatibilidad)

## 13. Exportación

Se generan dos archivos:

1. `conteos_censales_2012_agrupado.csv`: base procesada para la integración histórica.
2. `compatibilidad_variables_2012.csv`: registro de variables disponibles para documentar la homologación.

In [ ]:
SALIDA_DATOS = "/content/conteos_censales_2012_agrupado.csv"
SALIDA_COMPAT = "/content/compatibilidad_variables_2012.csv"

df_2012_agrupado.to_csv(
    SALIDA_DATOS,
    index=False,
    encoding="utf-8-sig"
)

compatibilidad.to_csv(
    SALIDA_COMPAT,
    index=False,
    encoding="utf-8-sig"
)

print("Archivos generados:")
print(" -", SALIDA_DATOS)
print(" -", SALIDA_COMPAT)

### Descarga opcional en Google Colab

Ejecuta la siguiente celda si quieres descargar inmediatamente los dos CSV a tu computadora.

In [ ]:
# Descomenta estas líneas si deseas descargar los archivos:
#
# from google.colab import files
# files.download(SALIDA_DATOS)
# files.download(SALIDA_COMPAT)

# Notas para el documento técnico

### Metodología aplicada a 2012

La base de los Conteos Censales de Participación Ciudadana 2012 se procesa mediante el mismo procedimiento utilizado en los años posteriores. Los registros se agrupan por entidad federativa, grupo de edad, sexo y tipo de sección electoral. Antes de calcular los indicadores de participación se suman los conteos de lista nominal, personas que sí votaron, personas que no votaron y, cuando se encuentra disponible, registros de participación no especificada.

La tasa de participación se calcula como:

$$
\frac{SV}{SV+NV}\times 100
$$

mientras que la abstención se calcula como:

$$
\frac{NV}{SV+NV}\times 100
$$

Los registros con tamaño determinado menor a 100 se conservan y se identifican mediante la variable `GRUPO_PEQUENO`, con el objetivo de evaluar posteriormente la sensibilidad del método de segmentación ante grupos de tamaño reducido.

> **Nota:** el umbral de 100 es un criterio operativo provisional del proyecto para el análisis de sensibilidad; no se interpreta como un criterio oficial del INE.

### Decisiones que deben mantenerse

| Decisión | Criterio |
|---|---|
| Unidad de análisis | Entidad + grupo de edad + sexo + tipo de sección |
| Grupos de edad | Mismo esquema utilizado en 2024 |
| Porcentajes | Calcular después de sumar los conteos |
| `NS` | Excluir del denominador de participación/abstención y conservar como indicador separado |
| Grupos pequeños | No eliminar; marcar con `N_DETERMINADO < 100` |
| Códigos desconocidos | Detener el pipeline y revisar el diccionario; no recodificar automáticamente |

### Próximo paso

Una vez validado 2012, se aplicará el mismo pipeline a 2009. Después se integrarán los seis años comparables para construir variables históricas de participación, tendencia y variabilidad antes de realizar el clustering.
